# Mask R-CNN Instance Segmentation — Dog & Cat Dataset

This notebook demonstrates training a **Mask R-CNN** model using the [Matterport Mask-RCNN](https://github.com/matterport/Mask_RCNN) implementation on a custom **Dog & Cat** dataset downloaded from Roboflow in COCO format.

**Workflow:**
1. Install Mask R-CNN dependencies
2. Download Dataset (via Roboflow)
3. Configure and train Mask R-CNN
4. Run inference on test images
5. Calculate evaluation metrics

> **Prerequisites:** Set your Roboflow API key as an environment variable:  
> `export ROBOFLOW_API_KEY="your_key_here"` (Linux/macOS)  
> `$env:ROBOFLOW_API_KEY = "your_key_here"` (Windows PowerShell)

In [ ]:
# ── Step 1: Install Dependencies ─────────────────────────────────────────────
# Install Matterport Mask R-CNN and required libraries
!pip install git+https://github.com/matterport/Mask_RCNN.git --quiet
!pip install roboflow scikit-learn Pillow --quiet

In [ ]:
# ── Step 2: Download Dataset from Roboflow (COCO format) ─────────────────────
# The API key is loaded from an environment variable to keep it secure.
# Set it before running: export ROBOFLOW_API_KEY="your_key_here"

import os
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "ROBOFLOW_API_KEY environment variable is not set. "
        "Please set it before running this cell."
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace("appledetect").project("dog_cat_objects")
version = project.version(1)
dataset = version.download("coco")  # Download in COCO format for Mask R-CNN

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# ── Step 3: Configure and Train Mask R-CNN ───────────────────────────────────
import mrcnn
from mrcnn import model as modellib
from mrcnn.config import Config
from mrcnn import utils
import os

# Model output directory (relative path, safe for all environments)
MODEL_DIR = os.path.join(os.getcwd(), "mrcnn_logs")
os.makedirs(MODEL_DIR, exist_ok=True)

# Configuration for the custom Dog & Cat dataset
class DogCatConfig(Config):
    """Configuration for training on the Dog & Cat dataset."""
    NAME = "dog_cat"
    IMAGES_PER_GPU = 1       # Reduce if running on CPU or low-memory GPU
    NUM_CLASSES = 1 + 2      # Background + cats + dogs
    STEPS_PER_EPOCH = 100
    DETECTION_MIN_CONFIDENCE = 0.9

config = DogCatConfig()
config.display()

# Initialize the model in training mode
model = modellib.MaskRCNN(mode="training", config=config, model_dir=MODEL_DIR)

# Load pre-trained COCO weights (transfer learning from heads only)
# Download from: https://github.com/matterport/Mask_RCNN/releases
COCO_WEIGHTS_PATH = "mask_rcnn_coco.h5"
model.load_weights(COCO_WEIGHTS_PATH, by_name=True, exclude=[
    "mrcnn_class_logits", "mrcnn_bbox_fc",
    "mrcnn_bbox", "mrcnn_mask"
])

# NOTE: You must create/load your dataset objects (train_dataset, val_dataset)
# using the COCO annotation files before calling model.train().
# See https://github.com/matterport/Mask_RCNN for dataset loading examples.

# Train the network heads only (faster convergence for fine-tuning)
model.train(
    train_dataset, val_dataset,
    learning_rate=config.LEARNING_RATE,
    epochs=30,
    layers="heads"
)

In [ ]:
# ── Step 4: Run Inference on a Test Image ────────────────────────────────────
import skimage.io
from mrcnn import visualize

CLASS_NAMES = ["BG", "cats", "dogs"]  # Background + 2 custom classes

# Switch model to inference mode
inference_config = DogCatConfig()
inference_model = modellib.MaskRCNN(
    mode="inference", config=inference_config, model_dir=MODEL_DIR
)

# Load the last saved weights
inference_model.load_weights(
    inference_model.find_last(), by_name=True
)

# Load a test image (update path to a valid image)
test_image_path = "test_image.jpg"
image = skimage.io.imread(test_image_path)

# Run object detection
results = inference_model.detect([image], verbose=1)
r = results[0]

# Visualize the results
visualize.display_instances(
    image, r["rois"], r["masks"], r["class_ids"],
    CLASS_NAMES, r["scores"]
)

In [ ]:
# ── Step 5: Calculate Evaluation Metrics ─────────────────────────────────────
# NOTE: y_true and y_pred should be lists of ground-truth and predicted class
# labels collected from running inference across the entire validation set.
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score
)

# Example: replace with your actual prediction loop results
# y_true = [...]  # Ground-truth labels
# y_pred = [...]  # Predicted labels from model.detect()

conf_matrix = confusion_matrix(y_true, y_pred)
precision   = precision_score(y_true, y_pred, average="weighted")
recall      = recall_score(y_true, y_pred, average="weighted")
f1          = f1_score(y_true, y_pred, average="weighted")

print(f"Confusion Matrix:\n{conf_matrix}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")